# 17e-fixed — Exact FIFA Annex C World Cup 2026 modal-order bracket simulator

This notebook does the full tournament projection correctly:

1. Simulate or lock group-stage results.
2. Rank each group from 1st to 4th.
3. Select the 8 best third-placed teams.
4. Use the exact FIFA Annex C lookup table to place third-placed teams.
5. Fill the official Round-of-32 bracket slots.
6. Simulate the knockout stage and estimate stage probabilities.
7. Build one coherent projected bracket with a projected champion.

Important:

- This version does **not** use the previous eligibility fallback.
- It requires the exact 495-row FIFA Annex C table.
- The notebook can build that table automatically from the official FIFA regulations PDF.
- If extraction fails, it stops rather than silently using an approximate bracket.


In [ ]:
# Cell 1. Configuration.

N_SIMULATIONS = 100000
RANDOM_SEED = 42
MAX_GOALS = 10

PREDICTIONS_PATH = (
    "data/processed/12_current_wc2026_paper_picks/"
    "current_match_predictions.csv"
)

ACTUAL_RESULTS_PATH = "data/inputs/actual_group_results.csv"

OUTPUT_DIR = (
    "data/processed/17e_fixed_exact_annex_c_modal_order_projection"
)

# Official FIFA regulations PDF containing Annex C.
FIFA_REGULATIONS_PDF_URL = (
    "https://digitalhub.fifa.com/m/636f5c9c6f29771f/"
    "original/FWC2026_regulations_EN.pdf"
)

PDF_CACHE_PATH = "data/inputs/FWC2026_regulations_EN.pdf"
ANNEX_C_CSV_PATH = "data/inputs/official_annex_c_2026.csv"

# If the actual-results file is missing, these known results are used.
USE_BUILT_IN_ACTUAL_RESULTS_FALLBACK = True

# Deterministic bracket projection method.
# This builds one coherent bracket from the most frequent complete
# group order in Monte Carlo, then uses official Annex C and model
# probabilities to pick knockout winners.
DETERMINISTIC_PROJECTION_METHOD = "modal_group_order_plus_exact_annex_c"


In [ ]:
# Cell 2. Imports and helpers.

from pathlib import Path
from collections import defaultdict, Counter
from itertools import combinations
import json
import math
import re
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(RANDOM_SEED)

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

input_dir = Path("data/inputs")
input_dir.mkdir(parents=True, exist_ok=True)


def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False, default=str)


def normalize_probs(values):
    arr = np.asarray(values, dtype=float)
    arr = np.clip(arr, 1e-10, 1.0)
    return arr / arr.sum()


def poisson_pmf(lam, max_goals=10):
    lam = float(np.clip(lam, 0.05, 8.0))
    values = np.arange(max_goals + 1)
    probs = np.exp(-lam) * np.power(lam, values)
    probs = probs / np.array([math.factorial(int(x)) for x in values])
    return probs / probs.sum()


def match_result_label(home_goals, away_goals):
    if home_goals > away_goals:
        return "home"
    if home_goals < away_goals:
        return "away"
    return "draw"


def points_for(goals_for, goals_against):
    if goals_for > goals_against:
        return 3
    if goals_for == goals_against:
        return 1
    return 0


def expected_score_from_elo(elo_a, elo_b):
    return 1.0 / (1.0 + 10 ** (-(elo_a - elo_b) / 400.0))


def combo_key(groups):
    return "".join(sorted(groups))

print("Setup OK.")


In [ ]:
# Cell 3. Load match predictions.

pred_path = Path(PREDICTIONS_PATH)

if not pred_path.exists():
    raise FileNotFoundError(
        "Run notebook 12 first or update PREDICTIONS_PATH."
    )

matches = pd.read_csv(pred_path)

matches["commence_time"] = pd.to_datetime(
    matches["commence_time"],
    utc=True,
    errors="coerce",
)

name_fixes = {
    "Czechia": "Czech Republic",
    "Ivory Coast": "Côte d'Ivoire",
    "Curacao": "Curaçao",
    "Bosnia and Herzegovina": "Bosnia & Herzegovina",
}

for col in ["home_team", "away_team"]:
    matches[col] = matches[col].replace(name_fixes)

required_cols = [
    "home_team",
    "away_team",
    "lambda_home",
    "lambda_away",
    "final_p_home",
    "final_p_draw",
    "final_p_away",
]

missing = [col for col in required_cols if col not in matches.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

matches = matches.dropna(
    subset=["commence_time", "home_team", "away_team"]
).copy()

matches = matches.sort_values("commence_time").reset_index(drop=True)

print("Matches loaded:", len(matches))
display(matches.head())


In [ ]:
# Cell 4. Groups.

GROUPS = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "B": ["Canada", "Bosnia & Herzegovina", "Qatar", "Switzerland"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Côte d'Ivoire", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

TEAM_TO_GROUP = {
    team: group
    for group, teams_in_group in GROUPS.items()
    for team in teams_in_group
}

teams = sorted(TEAM_TO_GROUP.keys())

matches["group"] = matches["home_team"].map(TEAM_TO_GROUP)

unknown = matches[matches["group"].isna()][
    ["home_team", "away_team"]
].drop_duplicates()

if len(unknown) > 0:
    raise ValueError(
        "Some match teams could not be mapped to groups. "
        "Check name normalization."
    )

print("Teams:", len(teams))
print("Groups:", len(GROUPS))


In [ ]:
# Cell 5. Build or load exact FIFA Annex C table.

ANNEX_COLUMNS = ["1A", "1B", "1D", "1E", "1G", "1I", "1K", "1L"]
ALL_GROUP_LETTERS = list("ABCDEFGHIJKL")


def download_pdf_if_needed():
    pdf_path = Path(PDF_CACHE_PATH)

    if pdf_path.exists():
        return pdf_path

    print("Downloading FIFA regulations PDF...")

    try:
        import requests
    except ImportError:
        raise ImportError(
            "Install requests or manually place the FIFA PDF at "
            f"{PDF_CACHE_PATH}."
        )

    response = requests.get(FIFA_REGULATIONS_PDF_URL, timeout=60)
    response.raise_for_status()

    pdf_path.parent.mkdir(parents=True, exist_ok=True)
    pdf_path.write_bytes(response.content)

    return pdf_path


def extract_annex_c_text_from_pdf(pdf_path):
    try:
        from pypdf import PdfReader
    except ImportError:
        raise ImportError(
            "Install pypdf first: pip install pypdf"
        )

    reader = PdfReader(str(pdf_path))

    # Annex C is near the end of the regulations PDF.
    # We extract a wide page range and validate the parsed result strictly.
    text_parts = []

    for page_index in range(78, min(len(reader.pages), 97)):
        text = reader.pages[page_index].extract_text() or ""
        text_parts.append(text)

    return "\n".join(text_parts)


def parse_annex_c_rows(text):
    # Normalize whitespace and keep one line per potential table row.
    text = text.replace("\u2011", "-").replace("\xa0", " ")
    lines = []

    for raw_line in text.splitlines():
        line = re.sub(r"\s+", " ", raw_line.strip())
        if line:
            lines.append(line)

    rows = []
    pattern = re.compile(
        r"^(\d{1,3})\s+"
        r"(3[A-L])\s+(3[A-L])\s+(3[A-L])\s+(3[A-L])\s+"
        r"(3[A-L])\s+(3[A-L])\s+(3[A-L])\s+(3[A-L])"
    )

    for line in lines:
        m = pattern.search(line)

        if not m:
            continue

        option = int(m.group(1))
        values = list(m.groups()[1:])

        rows.append({
            "option": option,
            **dict(zip(ANNEX_COLUMNS, values)),
        })

    df = pd.DataFrame(rows)

    if len(df) == 0:
        raise ValueError(
            "No Annex C rows were parsed from the PDF. "
            "Check pypdf extraction or provide official_annex_c_2026.csv."
        )

    df = df.drop_duplicates("option", keep="first")
    df = df.sort_values("option").reset_index(drop=True)

    return df


def validate_annex_c_table(df):
    required = ["option"] + ANNEX_COLUMNS
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(f"Annex C table is missing columns: {missing}")

    if len(df) != 495:
        raise ValueError(
            f"Annex C must have 495 rows, got {len(df)}. "
            "Do not run with a partial third-place table."
        )

    if sorted(df["option"].astype(int).tolist()) != list(range(1, 496)):
        raise ValueError("Annex C option numbers must be exactly 1..495.")

    combo_keys = []

    for _, row in df.iterrows():
        values = [row[c] for c in ANNEX_COLUMNS]
        groups = [v.replace("3", "") for v in values]

        if len(set(groups)) != 8:
            raise ValueError(
                f"Option {row['option']} does not contain 8 unique groups."
            )

        for slot, source in zip(ANNEX_COLUMNS, values):
            if not re.fullmatch(r"3[A-L]", str(source)):
                raise ValueError(
                    f"Invalid source {source} in option {row['option']}."
                )

        combo_keys.append(combo_key(groups))

    expected_combos = {
        combo_key(c)
        for c in combinations(ALL_GROUP_LETTERS, 8)
    }

    actual_combos = set(combo_keys)

    if actual_combos != expected_combos:
        missing = sorted(expected_combos - actual_combos)[:10]
        extra = sorted(actual_combos - expected_combos)[:10]
        raise ValueError(
            "Annex C combo coverage is not exact. "
            f"Missing sample: {missing}; extra sample: {extra}"
        )

    if len(combo_keys) != len(set(combo_keys)):
        raise ValueError("Annex C has duplicate combo keys.")

    out = df.copy()
    out["combo_key"] = combo_keys

    return out


def load_or_build_annex_c():
    csv_path = Path(ANNEX_C_CSV_PATH)

    if csv_path.exists():
        print("Loading Annex C CSV:", csv_path)
        df = pd.read_csv(csv_path)
        return validate_annex_c_table(df)

    pdf_path = download_pdf_if_needed()
    text = extract_annex_c_text_from_pdf(pdf_path)
    df = parse_annex_c_rows(text)
    df = validate_annex_c_table(df)

    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False)

    print("Annex C extracted and saved to:", csv_path)
    return df

annex_c = load_or_build_annex_c()
annex_c.to_csv(output_dir / "official_annex_c_2026.csv", index=False)

annex_map = {
    row["combo_key"]: {
        col: row[col]
        for col in ANNEX_COLUMNS
    }
    for _, row in annex_c.iterrows()
}

annex_validation_report = {
    "rows": int(len(annex_c)),
    "options": "1..495",
    "unique_combo_keys": int(annex_c["combo_key"].nunique()),
    "validation_status": "passed",
    "source": "FIFA Regulations for the FIFA World Cup 26, Annexe C",
}

save_json(
    output_dir / "annex_c_validation_report.json",
    annex_validation_report,
)

display(annex_c.head())
display(pd.DataFrame([annex_validation_report]))


In [ ]:
# Cell 6. Load actual completed group-stage results.


def standardize_actual_results(df):
    df = df.copy()

    for col in ["home_team", "away_team"]:
        df[col] = df[col].replace(name_fixes)

    df["home_goals"] = pd.to_numeric(
        df["home_goals"],
        errors="coerce",
    )

    df["away_goals"] = pd.to_numeric(
        df["away_goals"],
        errors="coerce",
    )

    df = df.dropna(
        subset=["home_team", "away_team", "home_goals", "away_goals"]
    ).copy()

    df["home_goals"] = df["home_goals"].astype(int)
    df["away_goals"] = df["away_goals"].astype(int)

    return df


built_in_actual = pd.DataFrame([
    {"group": "A", "home_team": "Mexico", "away_team": "South Africa", "home_goals": 2, "away_goals": 0},
    {"group": "A", "home_team": "South Korea", "away_team": "Czech Republic", "home_goals": 2, "away_goals": 1},
    {"group": "B", "home_team": "Canada", "away_team": "Bosnia & Herzegovina", "home_goals": 1, "away_goals": 1},
    {"group": "D", "home_team": "United States", "away_team": "Paraguay", "home_goals": 4, "away_goals": 1},
    {"group": "B", "home_team": "Qatar", "away_team": "Switzerland", "home_goals": 1, "away_goals": 1},
    {"group": "C", "home_team": "Brazil", "away_team": "Morocco", "home_goals": 1, "away_goals": 1},
    {"group": "C", "home_team": "Haiti", "away_team": "Scotland", "home_goals": 0, "away_goals": 1},
    {"group": "D", "home_team": "Australia", "away_team": "Turkey", "home_goals": 2, "away_goals": 0},
])

actual_frames = []
actual_sources = []

actual_path = Path(ACTUAL_RESULTS_PATH)

if actual_path.exists():
    actual_frames.append(
        standardize_actual_results(pd.read_csv(actual_path))
    )
    actual_sources.append(str(actual_path))
elif USE_BUILT_IN_ACTUAL_RESULTS_FALLBACK:
    actual_frames.append(standardize_actual_results(built_in_actual))
    actual_sources.append("built_in_fallback")

if actual_frames:
    actual = pd.concat(actual_frames, ignore_index=True, sort=False)
    actual = actual.drop_duplicates(
        ["home_team", "away_team"],
        keep="last",
    )
else:
    actual = pd.DataFrame(
        columns=["home_team", "away_team", "home_goals", "away_goals"]
    )

actual_map = {
    (row["home_team"], row["away_team"]): (
        int(row["home_goals"]),
        int(row["away_goals"]),
    )
    for _, row in actual.iterrows()
}

actual.to_csv(output_dir / "actual_results_used.csv", index=False)

# Write a template that the user can keep updating.
template = matches[["commence_time", "group", "home_team", "away_team"]].copy()
template["home_goals"] = np.nan
template["away_goals"] = np.nan
template.to_csv(input_dir / "actual_group_results_template.csv", index=False)

print("Actual results loaded:", len(actual_map))
print("Actual result sources:", actual_sources)
display(actual)


In [ ]:
# Cell 7. Team strength fallback for arbitrary knockout matchups.

team_elo_values = defaultdict(list)

if "elo_home_pre" in matches.columns and "elo_away_pre" in matches.columns:
    for _, row in matches.iterrows():
        team_elo_values[row["home_team"]].append(row["elo_home_pre"])
        team_elo_values[row["away_team"]].append(row["elo_away_pre"])

team_elo = {}

for team in teams:
    vals = [
        float(v)
        for v in team_elo_values.get(team, [])
        if not pd.isna(v)
    ]
    team_elo[team] = float(np.mean(vals)) if vals else 1500.0

team_strength = pd.DataFrame([
    {"team": team, "group": TEAM_TO_GROUP[team], "elo": team_elo[team]}
    for team in teams
]).sort_values("elo", ascending=False)

team_strength.to_csv(output_dir / "team_strength_estimates.csv", index=False)

display(team_strength.head(20))


In [ ]:
# Cell 8. Calibrated scoreline distribution.


def score_matrix_from_lambdas(lambda_home, lambda_away):
    home_probs = poisson_pmf(lambda_home, MAX_GOALS)
    away_probs = poisson_pmf(lambda_away, MAX_GOALS)
    mat = np.outer(home_probs, away_probs)
    return mat / mat.sum()


def conditional_score_distribution(mat, outcome_label):
    cond = mat.copy()

    for hg in range(cond.shape[0]):
        for ag in range(cond.shape[1]):
            if match_result_label(hg, ag) != outcome_label:
                cond[hg, ag] = 0.0

    total = cond.sum()

    if total <= 0:
        return mat / mat.sum()

    return cond / total


def calibrated_score_matrix(row):
    p_home, p_draw, p_away = normalize_probs([
        row["final_p_home"],
        row["final_p_draw"],
        row["final_p_away"],
    ])

    base = score_matrix_from_lambdas(row["lambda_home"], row["lambda_away"])

    home_cond = conditional_score_distribution(base, "home")
    draw_cond = conditional_score_distribution(base, "draw")
    away_cond = conditional_score_distribution(base, "away")

    mat = p_home * home_cond + p_draw * draw_cond + p_away * away_cond
    return mat / mat.sum()


def sample_score_from_matrix(mat):
    flat = mat.reshape(-1)
    idx = rng.choice(len(flat), p=flat)
    hg, ag = np.unravel_index(idx, mat.shape)
    return int(hg), int(ag)


def expected_match_contribution(row):
    key = (row["home_team"], row["away_team"])

    if key in actual_map:
        hg, ag = actual_map[key]
        return {
            "home_xp": points_for(hg, ag),
            "away_xp": points_for(ag, hg),
            "home_xgf": float(hg),
            "home_xga": float(ag),
            "away_xgf": float(ag),
            "away_xga": float(hg),
        }

    mat = calibrated_score_matrix(row)

    home_xp = 0.0
    away_xp = 0.0
    home_xgf = 0.0
    away_xgf = 0.0

    for hg in range(mat.shape[0]):
        for ag in range(mat.shape[1]):
            p = mat[hg, ag]
            home_xp += p * points_for(hg, ag)
            away_xp += p * points_for(ag, hg)
            home_xgf += p * hg
            away_xgf += p * ag

    return {
        "home_xp": home_xp,
        "away_xp": away_xp,
        "home_xgf": home_xgf,
        "home_xga": away_xgf,
        "away_xgf": away_xgf,
        "away_xga": home_xgf,
    }


def simulate_group_match(row):
    key = (row["home_team"], row["away_team"])

    if key in actual_map:
        return actual_map[key]

    mat = calibrated_score_matrix(row)
    return sample_score_from_matrix(mat)


In [ ]:
# Cell 9. Group table functions.


def empty_group_table(group):
    return {
        team: {
            "team": team,
            "group": group,
            "played": 0,
            "points": 0.0,
            "goals_for": 0.0,
            "goals_against": 0.0,
            "goal_difference": 0.0,
            "random_tiebreaker": rng.random(),
        }
        for team in GROUPS[group]
    }


def apply_match(table, home, away, home_goals, away_goals):
    table[home]["played"] += 1
    table[away]["played"] += 1

    table[home]["goals_for"] += home_goals
    table[home]["goals_against"] += away_goals
    table[home]["goal_difference"] += home_goals - away_goals
    table[home]["points"] += points_for(home_goals, away_goals)

    table[away]["goals_for"] += away_goals
    table[away]["goals_against"] += home_goals
    table[away]["goal_difference"] += away_goals - home_goals
    table[away]["points"] += points_for(away_goals, home_goals)


def apply_expected_match(table, home, away, contrib):
    table[home]["played"] += 1
    table[away]["played"] += 1

    table[home]["points"] += contrib["home_xp"]
    table[away]["points"] += contrib["away_xp"]

    table[home]["goals_for"] += contrib["home_xgf"]
    table[home]["goals_against"] += contrib["home_xga"]
    table[home]["goal_difference"] += (
        contrib["home_xgf"] - contrib["home_xga"]
    )

    table[away]["goals_for"] += contrib["away_xgf"]
    table[away]["goals_against"] += contrib["away_xga"]
    table[away]["goal_difference"] += (
        contrib["away_xgf"] - contrib["away_xga"]
    )


def rank_group_table(table):
    df = pd.DataFrame(list(table.values()))

    df = df.sort_values(
        ["points", "goal_difference", "goals_for", "random_tiebreaker"],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)

    df["group_rank"] = np.arange(1, len(df) + 1)
    return df


def rank_third_place_table(thirds):
    return thirds.sort_values(
        ["points", "goal_difference", "goals_for", "random_tiebreaker"],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)


def simulate_group_stage():
    group_tables = []

    for group in GROUPS:
        table = empty_group_table(group)
        group_matches = matches[matches["group"] == group]

        for _, row in group_matches.iterrows():
            hg, ag = simulate_group_match(row)
            apply_match(table, row["home_team"], row["away_team"], hg, ag)

        group_tables.append(rank_group_table(table))

    all_tables = pd.concat(group_tables, ignore_index=True)

    top_two = all_tables[all_tables["group_rank"] <= 2].copy()
    thirds = all_tables[all_tables["group_rank"] == 3].copy()
    best_thirds = rank_third_place_table(thirds).head(8).copy()

    qualifiers = pd.concat([top_two, best_thirds], ignore_index=True)
    qualifiers["qualified_via"] = np.where(
        qualifiers["group_rank"] <= 2,
        "top_two",
        "best_third",
    )

    return all_tables, qualifiers


def build_expected_group_stage():
    group_tables = []

    for group in GROUPS:
        table = empty_group_table(group)
        group_matches = matches[matches["group"] == group]

        for _, row in group_matches.iterrows():
            contrib = expected_match_contribution(row)
            apply_expected_match(
                table,
                row["home_team"],
                row["away_team"],
                contrib,
            )

        group_tables.append(rank_group_table(table))

    all_tables = pd.concat(group_tables, ignore_index=True)

    top_two = all_tables[all_tables["group_rank"] <= 2].copy()
    thirds = all_tables[all_tables["group_rank"] == 3].copy()
    best_thirds = rank_third_place_table(thirds).head(8).copy()

    qualifiers = pd.concat([top_two, best_thirds], ignore_index=True)
    qualifiers["qualified_via"] = np.where(
        qualifiers["group_rank"] <= 2,
        "top_two",
        "best_third",
    )

    return all_tables, qualifiers, rank_third_place_table(thirds)


In [ ]:
# Cell 10. Official bracket slots and exact Annex C resolver.

# Round of 32 slots.
# Third-place slots are keyed by the group winner they face in Annex C.
R32_FIXED = {
    "M73": ("2A", "2B"),
    "M74": ("1E", "3_FOR_1E"),
    "M75": ("1F", "2C"),
    "M76": ("1C", "2F"),
    "M77": ("1I", "3_FOR_1I"),
    "M78": ("2E", "2I"),
    "M79": ("1A", "3_FOR_1A"),
    "M80": ("1L", "3_FOR_1L"),
    "M81": ("1D", "3_FOR_1D"),
    "M82": ("1G", "3_FOR_1G"),
    "M83": ("2K", "2L"),
    "M84": ("1H", "2J"),
    "M85": ("1B", "3_FOR_1B"),
    "M86": ("1J", "2H"),
    "M87": ("1K", "3_FOR_1K"),
    "M88": ("2D", "2G"),
}

R16_PAIRS = [
    ("M89", "M74", "M77"),
    ("M90", "M73", "M75"),
    ("M91", "M76", "M78"),
    ("M92", "M79", "M80"),
    ("M93", "M83", "M84"),
    ("M94", "M81", "M82"),
    ("M95", "M86", "M88"),
    ("M96", "M85", "M87"),
]

QF_PAIRS = [
    ("M97", "M89", "M90"),
    ("M98", "M93", "M94"),
    ("M99", "M91", "M92"),
    ("M100", "M95", "M96"),
]

SF_PAIRS = [
    ("M101", "M97", "M98"),
    ("M102", "M99", "M100"),
]

FINAL_PAIR = ("M104", "M101", "M102")

# Map Annex C columns to the actual Round-of-32 match ids.
ANNEX_COLUMN_TO_MATCH_ID = {
    "1A": "M79",
    "1B": "M85",
    "1D": "M81",
    "1E": "M74",
    "1G": "M82",
    "1I": "M77",
    "1K": "M87",
    "1L": "M80",
}


def qualifier_lookup(qualifiers):
    out = {}

    for _, row in qualifiers.iterrows():
        group = row["group"]

        if row["group_rank"] == 1:
            out[f"1{group}"] = row["team"]

        if row["group_rank"] == 2:
            out[f"2{group}"] = row["team"]

        if row["group_rank"] == 3:
            out[f"3{group}"] = row["team"]

    return out


def exact_third_assignment_from_annex_c(qualifiers):
    third_groups = sorted(
        row["group"]
        for _, row in qualifiers.iterrows()
        if row["group_rank"] == 3
    )

    key = combo_key(third_groups)

    if key not in annex_map:
        raise ValueError(
            f"No official Annex C row found for third-place combo {key}."
        )

    # Example output:
    # {
    #   "M79": "3E",  # 1A faces 3E
    #   "M85": "3G",  # 1B faces 3G
    # }
    mapping = {}

    for annex_col, source in annex_map[key].items():
        match_id = ANNEX_COLUMN_TO_MATCH_ID[annex_col]
        mapping[match_id] = source

    return key, mapping


def resolve_r32_slot(slot, lookup, third_assignment_by_match, match_id):
    if slot.startswith("3_FOR_"):
        source = third_assignment_by_match[match_id]
        return lookup[source]

    return lookup[slot]


def validate_full_bracket(match_participants, match_winners):
    # Round of 32 has 32 unique teams.
    r32_teams = []

    for match_id in R32_FIXED:
        a, b = match_participants[match_id]
        r32_teams.extend([a, b])

    if len(r32_teams) != 32 or len(set(r32_teams)) != 32:
        raise ValueError("Round of 32 must contain 32 unique teams.")

    # Later-stage participants must equal previous winners.
    for match_id, prev_a, prev_b in R16_PAIRS:
        expected = {match_winners[prev_a], match_winners[prev_b]}
        actual = set(match_participants[match_id])
        if actual != expected:
            raise ValueError(f"Invalid R16 dependency for {match_id}.")

    for match_id, prev_a, prev_b in QF_PAIRS:
        expected = {match_winners[prev_a], match_winners[prev_b]}
        actual = set(match_participants[match_id])
        if actual != expected:
            raise ValueError(f"Invalid quarterfinal dependency for {match_id}.")

    for match_id, prev_a, prev_b in SF_PAIRS:
        expected = {match_winners[prev_a], match_winners[prev_b]}
        actual = set(match_participants[match_id])
        if actual != expected:
            raise ValueError(f"Invalid semifinal dependency for {match_id}.")

    match_id, prev_a, prev_b = FINAL_PAIR
    expected = {match_winners[prev_a], match_winners[prev_b]}
    actual = set(match_participants[match_id])
    if actual != expected:
        raise ValueError("Invalid final dependency.")

    return True


def knockout_win_probability(team_a, team_b):
    elo_a = team_elo.get(team_a, 1500.0)
    elo_b = team_elo.get(team_b, 1500.0)
    return float(expected_score_from_elo(elo_a, elo_b))


def pick_knockout_winner(team_a, team_b, deterministic=False):
    p_a = knockout_win_probability(team_a, team_b)

    if deterministic:
        return team_a if p_a >= 0.5 else team_b

    return team_a if rng.random() < p_a else team_b


def build_knockout_bracket(qualifiers, deterministic=False):
    lookup = qualifier_lookup(qualifiers)
    third_combo, third_assignment_by_match = exact_third_assignment_from_annex_c(
        qualifiers
    )

    match_participants = {}
    match_winners = {}

    # Round of 32.
    for match_id, (left_slot, right_slot) in R32_FIXED.items():
        team_a = resolve_r32_slot(
            left_slot,
            lookup,
            third_assignment_by_match,
            match_id,
        )
        team_b = resolve_r32_slot(
            right_slot,
            lookup,
            third_assignment_by_match,
            match_id,
        )
        winner = pick_knockout_winner(team_a, team_b, deterministic)

        match_participants[match_id] = (team_a, team_b)
        match_winners[match_id] = winner

    # Round of 16.
    for match_id, prev_a, prev_b in R16_PAIRS:
        team_a = match_winners[prev_a]
        team_b = match_winners[prev_b]
        winner = pick_knockout_winner(team_a, team_b, deterministic)

        match_participants[match_id] = (team_a, team_b)
        match_winners[match_id] = winner

    # Quarterfinals.
    for match_id, prev_a, prev_b in QF_PAIRS:
        team_a = match_winners[prev_a]
        team_b = match_winners[prev_b]
        winner = pick_knockout_winner(team_a, team_b, deterministic)

        match_participants[match_id] = (team_a, team_b)
        match_winners[match_id] = winner

    # Semifinals.
    for match_id, prev_a, prev_b in SF_PAIRS:
        team_a = match_winners[prev_a]
        team_b = match_winners[prev_b]
        winner = pick_knockout_winner(team_a, team_b, deterministic)

        match_participants[match_id] = (team_a, team_b)
        match_winners[match_id] = winner

    # Final.
    match_id, prev_a, prev_b = FINAL_PAIR
    team_a = match_winners[prev_a]
    team_b = match_winners[prev_b]
    winner = pick_knockout_winner(team_a, team_b, deterministic)

    match_participants[match_id] = (team_a, team_b)
    match_winners[match_id] = winner

    validate_full_bracket(match_participants, match_winners)

    stage_teams = {
        "round_of_32": set(sum(
            [list(match_participants[m]) for m in R32_FIXED],
            [],
        )),
        "round_of_16": {match_winners[m] for m in R32_FIXED},
        "quarterfinal": {match_winners[m[0]] for m in R16_PAIRS},
        "semifinal": {match_winners[m[0]] for m in QF_PAIRS},
        "final": {match_winners[m[0]] for m in SF_PAIRS},
        "champion": {winner},
    }

    return {
        "third_combo_key": third_combo,
        "third_assignment_by_match": third_assignment_by_match,
        "match_participants": match_participants,
        "match_winners": match_winners,
        "stage_teams": stage_teams,
        "champion": winner,
    }


In [ ]:
# Cell 11. Shared stage labels for bracket outputs.

# The single projected bracket is built after the Monte Carlo run.
# Reason:
# We need the most frequent full group ordering from simulations.
#
# This avoids ranking teams only by expected points, which can produce
# unintuitive edge cases when two teams are separated by a tiny expected
# points difference.

stage_by_match = {}

for match_id in R32_FIXED:
    stage_by_match[match_id] = "round_of_32"

for match_id, _, _ in R16_PAIRS:
    stage_by_match[match_id] = "round_of_16"

for match_id, _, _ in QF_PAIRS:
    stage_by_match[match_id] = "quarterfinal"

for match_id, _, _ in SF_PAIRS:
    stage_by_match[match_id] = "semifinal"

stage_by_match["M104"] = "final"
print("Stage labels ready.")


In [ ]:
# Cell 12. Monte Carlo simulation with exact Annex C.

stage_counts = {team: defaultdict(int) for team in teams}
group_counts = {team: defaultdict(int) for team in teams}
matchup_counts = defaultdict(Counter)
winner_counts = defaultdict(Counter)
third_combo_counts = Counter()

# New in 17e:
# Track the complete 1st/2nd/3rd/4th order for each group.
# We will build the single projected bracket from the most frequent
# complete group order, not from expected points.
group_order_counts = {group: Counter() for group in GROUPS}
group_order_stat_counts = Counter()
group_order_stat_sums = defaultdict(float)

example_runs = []

for sim_id in range(N_SIMULATIONS):
    group_table, qualifiers = simulate_group_stage()

    for _, row in group_table.iterrows():
        team = row["team"]

        if row["group_rank"] == 1:
            group_counts[team]["group_winner"] += 1

        if row["group_rank"] <= 2:
            group_counts[team]["top_two"] += 1

        if row["group_rank"] == 3:
            group_counts[team]["third_place"] += 1

    # Store complete group orders and conditional average stats.
    for group, part in group_table.groupby("group"):
        ordered = part.sort_values("group_rank").copy()
        order = tuple(ordered["team"].tolist())

        group_order_counts[group][order] += 1
        group_order_stat_counts[(group, order)] += 1

        for _, row in ordered.iterrows():
            team = row["team"]
            for col in [
                "played",
                "points",
                "goals_for",
                "goals_against",
                "goal_difference",
                "group_rank",
            ]:
                group_order_stat_sums[
                    (group, order, team, col)
                ] += float(row[col])

    knockout = build_knockout_bracket(
        qualifiers,
        deterministic=False,
    )

    third_combo_counts[knockout["third_combo_key"]] += 1

    for match_id, participants in knockout["match_participants"].items():
        matchup_counts[match_id][participants] += 1

    for match_id, winner in knockout["match_winners"].items():
        winner_counts[match_id][winner] += 1

    for stage, stage_team_set in knockout["stage_teams"].items():
        for team in stage_team_set:
            stage_counts[team][stage] += 1

    if sim_id < 5:
        example_runs.append({
            "simulation_id": sim_id,
            "group_table": group_table.to_dict("records"),
            "qualifiers": qualifiers.to_dict("records"),
            "third_combo_key": knockout["third_combo_key"],
            "third_assignment_by_match": knockout[
                "third_assignment_by_match"
            ],
            "match_participants": {
                k: list(v)
                for k, v in knockout["match_participants"].items()
            },
            "match_winners": knockout["match_winners"],
            "champion": knockout["champion"],
        })

    if (sim_id + 1) % 10000 == 0:
        print("Completed simulations:", sim_id + 1)

print("Done:", N_SIMULATIONS)


In [ ]:
# Cell 12b. Build one coherent projected bracket from modal group orders.

def build_modal_group_order_projection():
    rows = []

    for group in GROUPS:
        if not group_order_counts[group]:
            raise ValueError(f"No simulated group orders for group {group}.")

        modal_order, modal_count = group_order_counts[
            group
        ].most_common(1)[0]

        denom = group_order_stat_counts[(group, modal_order)]

        for team in modal_order:
            row = {
                "team": team,
                "group": group,
                "modal_group_order": " > ".join(modal_order),
                "modal_group_order_probability": (
                    modal_count / N_SIMULATIONS
                ),
            }

            for col in [
                "played",
                "points",
                "goals_for",
                "goals_against",
                "goal_difference",
                "group_rank",
            ]:
                row[col] = (
                    group_order_stat_sums[
                        (group, modal_order, team, col)
                    ] / denom
                )

            row["group_rank"] = int(round(row["group_rank"]))

            # Deterministic fallback for best-third ranking.
            # The original simulated random tiebreaker is not meaningful
            # for a single modal-order projection. We use Elo only after
            # points, goal difference, and goals for.
            row["random_tiebreaker"] = (
                team_elo.get(team, 1500.0) / 10000.0
            )

            rows.append(row)

    modal_group_table = pd.DataFrame(rows)

    # Keep ranks exactly 1..4 within each group according to modal order.
    modal_group_table = modal_group_table.sort_values(
        ["group", "group_rank"]
    ).reset_index(drop=True)

    top_two = modal_group_table[
        modal_group_table["group_rank"] <= 2
    ].copy()

    thirds = modal_group_table[
        modal_group_table["group_rank"] == 3
    ].copy()

    best_thirds = rank_third_place_table(thirds).head(8).copy()

    qualifiers = pd.concat([top_two, best_thirds], ignore_index=True)

    qualifiers["qualified_via"] = np.where(
        qualifiers["group_rank"] <= 2,
        "top_two",
        "best_third",
    )

    all_thirds_ranked = rank_third_place_table(thirds)

    return modal_group_table, qualifiers, all_thirds_ranked


modal_group_table, modal_qualifiers, modal_thirds = (
    build_modal_group_order_projection()
)

projected_knockout = build_knockout_bracket(
    modal_qualifiers,
    deterministic=True,
)

modal_group_table.to_csv(
    output_dir / "projected_modal_group_order_tables.csv",
    index=False,
)

modal_thirds.to_csv(
    output_dir / "projected_modal_third_place_ranking.csv",
    index=False,
)

modal_qualifiers.to_csv(
    output_dir / "projected_modal_qualified_teams.csv",
    index=False,
)

projected_rows = []

for match_id in sorted(
    projected_knockout["match_participants"].keys(),
    key=lambda x: int(x.replace("M", "")),
):
    team_a, team_b = projected_knockout["match_participants"][match_id]
    winner = projected_knockout["match_winners"][match_id]
    p_a = knockout_win_probability(team_a, team_b)

    projected_rows.append({
        "match_id": match_id,
        "stage": stage_by_match[match_id],
        "team_a": team_a,
        "team_b": team_b,
        "p_team_a": p_a,
        "p_team_b": 1.0 - p_a,
        "projected_winner": winner,
    })

single_projected_bracket = pd.DataFrame(projected_rows)

single_projected_bracket.to_csv(
    output_dir / "single_projected_official_bracket.csv",
    index=False,
)

lookup = qualifier_lookup(modal_qualifiers)

third_assignment_rows = [
    {
        "match_id": match_id,
        "annex_source": source,
        "team": lookup[source],
    }
    for match_id, source in projected_knockout[
        "third_assignment_by_match"
    ].items()
]

projected_annex_c_assignment = pd.DataFrame(third_assignment_rows)

projected_annex_c_assignment.to_csv(
    output_dir / "projected_annex_c_third_place_assignment.csv",
    index=False,
)

projected_bracket_validation = {
    "validation_status": "passed",
    "projection_method": DETERMINISTIC_PROJECTION_METHOD,
    "third_combo_key": projected_knockout["third_combo_key"],
    "annex_c_row_exists": projected_knockout[
        "third_combo_key"
    ] in annex_map,
    "unique_round_of_32_teams": int(
        len(set(
            single_projected_bracket[
                single_projected_bracket["stage"] == "round_of_32"
            ][["team_a", "team_b"]].values.ravel()
        ))
    ),
    "projected_champion": projected_knockout["champion"],
    "important_note": (
        "This is one coherent bracket built from the most frequent "
        "complete order of each group across Monte Carlo simulations. "
        "Third-placed teams are placed through exact FIFA Annex C."
    ),
}

if projected_bracket_validation["unique_round_of_32_teams"] != 32:
    raise ValueError("Projected bracket does not contain 32 unique teams.")

if not projected_bracket_validation["annex_c_row_exists"]:
    raise ValueError("Projected third-place combo is not in Annex C.")

save_json(
    output_dir / "single_projected_bracket_validation.json",
    projected_bracket_validation,
)

display(modal_group_table)
display(modal_thirds)
display(single_projected_bracket)
display(pd.DataFrame([projected_bracket_validation]))


In [ ]:
# Cell 13. Probability outputs.

rows = []

for team in teams:
    rows.append({
        "team": team,
        "group": TEAM_TO_GROUP[team],
        "elo": team_elo.get(team, 1500.0),
        "group_winner_probability": group_counts[team]["group_winner"] / N_SIMULATIONS,
        "top_two_probability": group_counts[team]["top_two"] / N_SIMULATIONS,
        "third_place_probability": group_counts[team]["third_place"] / N_SIMULATIONS,
        "round_of_32_probability": stage_counts[team]["round_of_32"] / N_SIMULATIONS,
        "round_of_16_probability": stage_counts[team]["round_of_16"] / N_SIMULATIONS,
        "quarterfinal_probability": stage_counts[team]["quarterfinal"] / N_SIMULATIONS,
        "semifinal_probability": stage_counts[team]["semifinal"] / N_SIMULATIONS,
        "final_probability": stage_counts[team]["final"] / N_SIMULATIONS,
        "champion_probability": stage_counts[team]["champion"] / N_SIMULATIONS,
    })

probabilities = pd.DataFrame(rows).sort_values(
    "champion_probability",
    ascending=False,
).reset_index(drop=True)

probability_cols = [
    c for c in probabilities.columns
    if c.endswith("_probability")
]

max_probability = float(probabilities[probability_cols].max().max())

if max_probability > 1.000001:
    raise ValueError(f"Probability above 1 detected: {max_probability}")

probabilities.to_csv(
    output_dir / "tournament_stage_probabilities.csv",
    index=False,
)

group_advancement = probabilities.sort_values(
    ["group", "round_of_32_probability", "champion_probability"],
    ascending=[True, False, False],
).reset_index(drop=True)

group_advancement.to_csv(
    output_dir / "group_advancement_probabilities.csv",
    index=False,
)

save_json(
    output_dir / "example_simulation_runs.json",
    example_runs,
)

save_json(
    output_dir / "third_place_combo_counts.json",
    dict(third_combo_counts.most_common(30)),
)

display(probabilities.head(20))
print("Max probability:", max_probability)


In [ ]:
# Cell 14. Modal match-slot diagnostics.

# This is diagnostic only. It should not be presented as one coherent bracket.
# Use single_projected_official_bracket.csv for the coherent projected bracket.

modal_rows = []

for match_id in sorted(
    matchup_counts.keys(),
    key=lambda x: int(x.replace("M", "")),
):
    matchup, matchup_n = matchup_counts[match_id].most_common(1)[0]
    winner, winner_n = winner_counts[match_id].most_common(1)[0]

    modal_rows.append({
        "match_id": match_id,
        "stage": stage_by_match[match_id],
        "team_a": matchup[0],
        "team_b": matchup[1],
        "modal_matchup_probability": matchup_n / N_SIMULATIONS,
        "modal_slot_winner": winner,
        "modal_slot_winner_probability": winner_n / N_SIMULATIONS,
    })

modal_match_slot_diagnostics = pd.DataFrame(modal_rows)
modal_match_slot_diagnostics.to_csv(
    output_dir / "modal_match_slot_diagnostics.csv",
    index=False,
)

display(modal_match_slot_diagnostics.head(20))


In [ ]:
# Cell 15. Charts.

top_champions = probabilities.head(15).copy()

plt.figure(figsize=(10, 7))
plt.barh(
    top_champions["team"][::-1],
    top_champions["champion_probability"][::-1],
)
plt.xlabel("Champion probability")
plt.title("Top 15 champion probabilities")
plt.tight_layout()
plt.savefig(
    output_dir / "top_15_champion_probabilities.png",
    dpi=160,
)
plt.show()

top_finalists = probabilities.sort_values(
    "final_probability",
    ascending=False,
).head(15)

plt.figure(figsize=(10, 7))
plt.barh(
    top_finalists["team"][::-1],
    top_finalists["final_probability"][::-1],
)
plt.xlabel("Final probability")
plt.title("Top 15 final probabilities")
plt.tight_layout()
plt.savefig(
    output_dir / "top_15_final_probabilities.png",
    dpi=160,
)
plt.show()


In [ ]:
# Cell 16. Final report bundle.

summary = {
    "n_simulations": N_SIMULATIONS,
    "random_seed": RANDOM_SEED,
    "matches_loaded": int(len(matches)),
    "actual_results_loaded": int(len(actual_map)),
    "actual_result_sources": actual_sources,
    "teams": int(len(teams)),
    "groups": int(len(GROUPS)),
    "annex_c_rows": int(len(annex_c)),
    "annex_c_validation": "passed",
    "max_probability": max_probability,
    "group_match_simulation": (
        "Calibrated score matrix: final 1X2 probabilities combined "
        "with Poisson scoreline distributions conditional on outcome."
    ),
    "knockout_method": (
        "Exact FIFA Annex C lookup for third-placed teams, official "
        "Round-of-32 slots, and official later-round winner dependencies."
    ),
    "projection_method": DETERMINISTIC_PROJECTION_METHOD,
    "projected_champion_single_bracket": projected_knockout["champion"],
    "single_bracket_source": "modal_complete_group_orders",
    "top_champion_by_monte_carlo": probabilities.iloc[0]["team"],
    "top_champion_probability": float(probabilities.iloc[0]["champion_probability"]),
}

save_json(output_dir / "summary.json", summary)

zip_path = (
    Path("data/processed")
    / "17e_fixed_exact_annex_c_modal_order_projection_report_bundle.zip"
)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in output_dir.rglob("*"):
        if file.is_file():
            zf.write(file, arcname=file.relative_to(output_dir))

display(pd.DataFrame([summary]))
print("Created:", zip_path.resolve())
